In [40]:
import torch
import pandas as pd
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [4]:
df = pd.read_csv('air.csv')
df.head()

,PM2.5,PM10,NOx,NO2,SO2,VOCs,CO,CO2,CH4,Temperature
0,39.967142,57.926035,116.192213,55.230299,4.531693,75.317261,2.789606,427.674347,1.706105,31.085120
1,101.935672,150.774299,76.826826,79.051618,18.744780,145.083987,1.966569,529.739619,2.492663,33.711103
2,70.996192,138.948796,158.731020,60.466604,14.892239,145.147338,2.626446,499.889443,2.431165,33.778698
3,28.464728,63.643900,25.385343,15.333286,7.647429,130.022319,1.779360,388.283712,1.818563,31.565877
4,78.265276,113.977926,105.644340,59.202337,17.696806,181.713667,3.240533,464.739197,2.597225,32.229835


In [9]:
x=df.drop(columns='Temperature')
y = df[['Temperature']].values
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=.2)

In [24]:
x_scaled = StandardScaler().fit(x_train)
y_scaled = StandardScaler().fit(y_train)

In [28]:
T = lambda a: torch.tensor(a, dtype=torch.float32)

x_train_t, x_test_t = (T(x_scaled.transform(a)) for a in (x_train, x_test))
y_train_t, y_test_t = (T(y_scaled.transform(a)) for a in (y_train, y_test))

In [63]:
class neu(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.layers= nn.Sequential(
            nn.Linear(n_in, 16), nn.ReLU(),
            nn.Linear(16,8), nn.ReLU(),
            nn.Linear(8,1)
        )
        
    def forward(self,x):
        return self.layers(x)

In [41]:
train_loader = DataLoader(TensorDataset(x_train_t, y_train_t), batch_size=32, shuffle=True)

In [76]:
model = neu(x_train_t.shape[1])
criterion = nn.MSELoss()
opt = optim.Adam(model.parameters(), lr=.0001)

In [77]:
epochs = 200

for epoch in range(epochs):
    model.train()
    for xb, yb in train_loader:
        opt.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        opt.step()
    if (epoch+1)%10==0:
        print(f'loss: {loss.item()}')

loss: 1.4182920455932617
loss: 1.4639472961425781
loss: 0.5239204168319702
loss: 1.000148057937622
loss: 1.054175615310669
loss: 0.5841670036315918
loss: 0.6215717196464539
loss: 1.3325374126434326
loss: 1.0287426710128784
loss: 1.0159058570861816
loss: 0.9704025983810425
loss: 1.6275057792663574
loss: 0.5354870557785034
loss: 0.5779334306716919
loss: 0.3918897807598114
loss: 0.5846577286720276
loss: 1.1963286399841309
loss: 0.601874589920044
loss: 0.48008912801742554
loss: 0.6272242069244385


In [78]:
model.eval()
with torch.no_grad():
    test_pred = model(x_test_t)
    test_loss = criterion(test_pred, y_test_t)

print(f'test loss: {test_loss.item()}')

test loss: 0.847153902053833
